# Consistency Check: DadaGP Tokens vs Parsed Events

This notebook verifies consistency between:
1. **Original DadaGP tokens** (raw .tokens.txt format)
2. **Parsed events** (from `dadagp_parser.py`)

We'll check:
- Token count matches
- Note events match (string, fret, pitch)
- Timing matches (wait tokens → TIME_SHIFT events)
- Metadata matches (tempo, downtune, etc.)
- Round-trip conversion (events → tokens → events)

In [1]:
import sys
from pathlib import Path
from collections import Counter
import random

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from src.dadagp_parser import (
    parse_dadagp_file,
    parse_dadagp_file_to_events,
    NoteToken, WaitToken, MetadataToken, StructuralToken,
    NoteOnEvent, NoteOffEvent, TimeShiftEvent, TabEvent,
    compute_pitch, STANDARD_TUNING
)

print("✓ Imports successful")

✓ Imports successful


## 1. Load Sample Files

In [2]:
# Find token files
dadagp_dir = Path('../DadaGP-v1.1')
token_files = list(dadagp_dir.rglob('*.tokens.txt'))

print(f"Found {len(token_files):,} token files")

if token_files:
    # Select sample files
    sample_files = random.sample(token_files, min(3, len(token_files)))
    print(f"\nSelected {len(sample_files)} files for consistency check:")
    for i, f in enumerate(sample_files, 1):
        print(f"  {i}. {f.relative_to(dadagp_dir)}")
else:
    print("No token files found")
    sample_files = []

Found 26,181 token files

Selected 3 files for consistency check:
  1. R/Rhapsody/Rhapsody - When Demons Awake (solo).gp3.tokens.txt
  2. K/Knopfler, Mark/Knopfler, Mark - Sailing To Philadelphia.gp3.tokens.txt
  3. G/Guitar Player Licoes/Guitar Player Lia%0es - Licks de Frank Gambale.gp3.tokens.txt


## 2. Parse File with Both Methods

In [3]:
if sample_files:
    sample_file = sample_files[0]
    print(f"Analyzing: {sample_file.name}")
    print("="*80)
    
    # Method 1: Parse raw tokens
    raw_tokens = parse_dadagp_file(str(sample_file))
    
    # Method 2: Parse to events
    input_events, output_events = parse_dadagp_file_to_events(str(sample_file))
    
    print(f"Raw tokens:    {len(raw_tokens):,}")
    print(f"Input events:  {len(input_events):,}")
    print(f"Output events: {len(output_events):,}")
else:
    raw_tokens = []
    input_events = []
    output_events = []

Analyzing: Rhapsody - When Demons Awake (solo).gp3.tokens.txt
Raw tokens:    394
Input events:  499
Output events: 665


## 3. Count Token Types

In [4]:
if raw_tokens:
    print("Raw token type distribution:")
    print("-"*80)
    
    # Count by type
    raw_counts = Counter([t.type for t in raw_tokens])
    for token_type, count in raw_counts.most_common():
        print(f"  {token_type:15s}: {count:5,}")
    
    # Count specific types
    note_tokens = [t for t in raw_tokens if isinstance(t, NoteToken)]
    wait_tokens = [t for t in raw_tokens if isinstance(t, WaitToken)]
    metadata_tokens = [t for t in raw_tokens if isinstance(t, MetadataToken)]
    structural_tokens = [t for t in raw_tokens if isinstance(t, StructuralToken)]
    
    print(f"\nDetailed counts:")
    print(f"  Note tokens:       {len(note_tokens):5,}")
    print(f"  Wait tokens:       {len(wait_tokens):5,}")
    print(f"  Metadata tokens:   {len(metadata_tokens):5,}")
    print(f"  Structural tokens: {len(structural_tokens):5,}")

Raw token type distribution:
--------------------------------------------------------------------------------
  wait           :   167
  note           :   166
  effect         :    43
  structural     :    15
  metadata       :     3

Detailed counts:
  Note tokens:         166
  Wait tokens:         167
  Metadata tokens:       3
  Structural tokens:    15


## 4. Verify Note Count Consistency

In [5]:
if raw_tokens and input_events:
    print("Note count consistency check:")
    print("="*80)
    
    # Count notes from raw tokens
    raw_note_count = len([t for t in raw_tokens if isinstance(t, NoteToken)])
    
    # Count NOTE_ON events (each note should produce one NOTE_ON)
    note_on_count = len([e for e in input_events if isinstance(e, NoteOnEvent)])
    
    # Count NOTE_OFF events
    note_off_count = len([e for e in input_events if isinstance(e, NoteOffEvent)])
    
    # Count TAB events
    tab_count = len([e for e in output_events if isinstance(e, TabEvent)])
    
    print(f"Raw note tokens:  {raw_note_count:,}")
    print(f"NOTE_ON events:   {note_on_count:,}")
    print(f"NOTE_OFF events:  {note_off_count:,}")
    print(f"TAB events:       {tab_count:,}")
    
    # Verify consistency
    checks = []
    
    if raw_note_count == note_on_count:
        print("\n✓ PASS: Raw notes == NOTE_ON events")
        checks.append(True)
    else:
        print(f"\n✗ FAIL: Raw notes ({raw_note_count}) != NOTE_ON events ({note_on_count})")
        checks.append(False)
    
    if note_on_count == note_off_count:
        print("✓ PASS: NOTE_ON == NOTE_OFF (all notes closed)")
        checks.append(True)
    else:
        print(f"✗ FAIL: NOTE_ON ({note_on_count}) != NOTE_OFF ({note_off_count})")
        checks.append(False)
    
    if note_on_count == tab_count:
        print("✓ PASS: NOTE_ON == TAB events")
        checks.append(True)
    else:
        print(f"✗ FAIL: NOTE_ON ({note_on_count}) != TAB events ({tab_count})")
        checks.append(False)
    
    if all(checks):
        print("\n" + "="*80)
        print("✓ ALL NOTE COUNTS CONSISTENT")
        print("="*80)
    else:
        print("\n" + "="*80)
        print("✗ INCONSISTENCY DETECTED")
        print("="*80)

Note count consistency check:
Raw note tokens:  166
NOTE_ON events:   166
NOTE_OFF events:  166
TAB events:       166

✓ PASS: Raw notes == NOTE_ON events
✓ PASS: NOTE_ON == NOTE_OFF (all notes closed)
✓ PASS: NOTE_ON == TAB events

✓ ALL NOTE COUNTS CONSISTENT


## 5. Verify Timing Consistency

In [6]:
if raw_tokens and input_events:
    print("Timing consistency check:")
    print("="*80)
    
    # Count wait tokens
    wait_tokens = [t for t in raw_tokens if isinstance(t, WaitToken)]
    total_wait_ticks = sum(t.ticks for t in wait_tokens)
    
    # Count TIME_SHIFT events in input
    time_shifts_input = [e for e in input_events if isinstance(e, TimeShiftEvent)]
    total_shift_ticks_input = sum(e.delta for e in time_shifts_input)
    
    # Count TIME_SHIFT events in output
    time_shifts_output = [e for e in output_events if isinstance(e, TimeShiftEvent)]
    total_shift_ticks_output = sum(e.delta for e in time_shifts_output)
    
    print(f"Wait tokens:           {len(wait_tokens):,}")
    print(f"Total wait ticks:      {total_wait_ticks:,}")
    print(f"\nTIME_SHIFT (input):    {len(time_shifts_input):,}")
    print(f"Total shift ticks:     {total_shift_ticks_input:,}")
    print(f"\nTIME_SHIFT (output):   {len(time_shifts_output):,}")
    print(f"Total shift ticks:     {total_shift_ticks_output:,}")
    
    # Verify consistency
    if total_wait_ticks == total_shift_ticks_input == total_shift_ticks_output:
        print("\n✓ PASS: Total timing matches across all representations")
    else:
        print("\n✗ FAIL: Timing mismatch:")
        if total_wait_ticks != total_shift_ticks_input:
            print(f"  Wait ticks ({total_wait_ticks}) != Input shifts ({total_shift_ticks_input})")
        if total_shift_ticks_input != total_shift_ticks_output:
            print(f"  Input shifts ({total_shift_ticks_input}) != Output shifts ({total_shift_ticks_output})")

Timing consistency check:
Wait tokens:           167
Total wait ticks:      49,920

TIME_SHIFT (input):    167
Total shift ticks:     49,920

TIME_SHIFT (output):   167
Total shift ticks:     49,920

✓ PASS: Total timing matches across all representations


## 6. Verify Note Details (String, Fret, Pitch)

In [7]:
if raw_tokens and input_events and output_events:
    print("Note detail verification (first 20 notes):")
    print("="*80)
    
    # Extract note tokens
    note_tokens = [t for t in raw_tokens if isinstance(t, NoteToken)]
    
    # Extract NOTE_ON events
    note_on_events = [e for e in input_events if isinstance(e, NoteOnEvent)]
    
    # Extract TAB events
    tab_events = [e for e in output_events if isinstance(e, TabEvent)]
    
    # Get metadata for pitch calculation
    metadata_tokens = [t for t in raw_tokens if isinstance(t, MetadataToken)]
    downtune = 0
    for t in metadata_tokens:
        if t.key == 'downtune':
            downtune = int(t.value)
    
    print(f"{'#':<4} {'Raw Token':<25} {'Expected Pitch':<15} {'Actual Pitch':<13} {'TAB':<15} {'Match':<6}")
    print("-"*80)
    
    mismatches = 0
    for i in range(min(20, len(note_tokens))):
        raw = note_tokens[i]
        note_on = note_on_events[i] if i < len(note_on_events) else None
        tab = tab_events[i] if i < len(tab_events) else None
        
        # Calculate expected pitch from raw token
        expected_pitch = compute_pitch(raw.string, raw.fret, downtune)
        
        # Get actual pitch from NOTE_ON event
        actual_pitch = note_on.pitch if note_on else None
        
        # Get TAB info
        tab_str = f"s{tab.string} f{tab.fret}" if tab else "N/A"
        
        # Check match
        pitch_match = (expected_pitch == actual_pitch) if actual_pitch else False
        tab_match = (raw.string == tab.string and raw.fret == tab.fret) if tab else False
        match = "✓" if (pitch_match and tab_match) else "✗"
        
        if not (pitch_match and tab_match):
            mismatches += 1
        
        raw_str = f"s{raw.string} f{raw.fret}"
        print(f"{i+1:<4} {raw_str:<25} {expected_pitch:<15} {actual_pitch:<13} {tab_str:<15} {match:<6}")
    
    if mismatches == 0:
        print("\n✓ PASS: All note details match")
    else:
        print(f"\n✗ FAIL: {mismatches} mismatches found")

Note detail verification (first 20 notes):
#    Raw Token                 Expected Pitch  Actual Pitch  TAB             Match 
--------------------------------------------------------------------------------
1    s1 f12                    52              52            s1 f12          ✓     
2    s1 f17                    57              57            s1 f17          ✓     
3    s1 f16                    56              56            s1 f16          ✓     
4    s1 f17                    57              57            s1 f17          ✓     
5    s2 f13                    58              58            s2 f13          ✓     
6    s2 f17                    62              62            s2 f17          ✓     
7    s2 f15                    60              60            s2 f15          ✓     
8    s2 f17                    62              62            s2 f17          ✓     
9    s2 f15                    60              60            s2 f15          ✓     
10   s2 f18                    63   

## 7. Verify Sequence Alignment

In [8]:
if input_events and output_events:
    print("Sequence alignment check:")
    print("="*80)
    
    # Extract non-TAB events from output
    output_no_tab = [e for e in output_events if not isinstance(e, TabEvent)]
    
    # Compare types and values
    print(f"Input sequence length:       {len(input_events):,}")
    print(f"Output sequence length:      {len(output_events):,}")
    print(f"Output (no TAB) length:      {len(output_no_tab):,}")
    
    if len(input_events) == len(output_no_tab):
        print("\n✓ PASS: Input and output (no TAB) have same length")
        
        # Check if events match
        mismatches = 0
        for i, (inp, out) in enumerate(zip(input_events, output_no_tab)):
            if type(inp) != type(out):
                if mismatches < 5:  # Show first 5 mismatches
                    print(f"  Mismatch at {i}: {type(inp).__name__} vs {type(out).__name__}")
                mismatches += 1
            elif isinstance(inp, NoteOnEvent) and inp.pitch != out.pitch:
                if mismatches < 5:
                    print(f"  Mismatch at {i}: pitch {inp.pitch} vs {out.pitch}")
                mismatches += 1
            elif isinstance(inp, TimeShiftEvent) and inp.delta != out.delta:
                if mismatches < 5:
                    print(f"  Mismatch at {i}: delta {inp.delta} vs {out.delta}")
                mismatches += 1
        
        if mismatches == 0:
            print("✓ PASS: All events match perfectly")
            print("\n✓ ALIGNMENT VERIFIED: Output = Input + TAB tokens")
        else:
            print(f"\n✗ FAIL: {mismatches} event mismatches found")
    else:
        print(f"\n✗ FAIL: Length mismatch (diff: {abs(len(input_events) - len(output_no_tab))})")

Sequence alignment check:
Input sequence length:       499
Output sequence length:      665
Output (no TAB) length:      499

✓ PASS: Input and output (no TAB) have same length
✓ PASS: All events match perfectly

✓ ALIGNMENT VERIFIED: Output = Input + TAB tokens


## 8. Metadata Consistency

In [9]:
if raw_tokens:
    print("Metadata extraction:")
    print("="*80)
    
    metadata = {}
    for token in raw_tokens:
        if isinstance(token, MetadataToken):
            metadata[token.key] = token.value
    
    print("Extracted metadata:")
    for key, value in metadata.items():
        print(f"  {key:15s}: {value}")
    
    # Verify these were used in pitch calculation
    if 'downtune' in metadata:
        downtune = int(metadata['downtune'])
        print(f"\n✓ Downtune value ({downtune}) used for pitch calculation")
    
    if 'tempo' in metadata:
        print(f"✓ Tempo value ({metadata['tempo']}) available for timing")

Metadata extraction:
Extracted metadata:
  artist         : rhapsody
  downtune       : 0
  tempo          : 160

✓ Downtune value (0) used for pitch calculation
✓ Tempo value (160) available for timing


## 9. Multi-File Consistency Check

In [10]:
if sample_files:
    print("Multi-file consistency check:")
    print("="*80)
    
    results = []
    
    for f in sample_files:
        try:
            # Parse both ways
            raw = parse_dadagp_file(str(f))
            inp, out = parse_dadagp_file_to_events(str(f))
            
            # Count notes
            raw_notes = len([t for t in raw if isinstance(t, NoteToken)])
            note_ons = len([e for e in inp if isinstance(e, NoteOnEvent)])
            tabs = len([e for e in out if isinstance(e, TabEvent)])
            
            # Count timing
            raw_waits = [t for t in raw if isinstance(t, WaitToken)]
            total_wait = sum(t.ticks for t in raw_waits)
            inp_shifts = [e for e in inp if isinstance(e, TimeShiftEvent)]
            total_shift = sum(e.delta for e in inp_shifts)
            
            # Check consistency
            notes_ok = (raw_notes == note_ons == tabs)
            timing_ok = (total_wait == total_shift)
            
            results.append({
                'file': f.name,
                'notes_ok': notes_ok,
                'timing_ok': timing_ok,
                'raw_notes': raw_notes,
                'note_ons': note_ons,
                'tabs': tabs
            })
        except Exception as e:
            print(f"  Error processing {f.name}: {e}")
            results.append({
                'file': f.name,
                'notes_ok': False,
                'timing_ok': False,
                'error': str(e)
            })
    
    # Display results
    print(f"\n{'File':<45} {'Notes':>8} {'Timing':>8} {'Status':>10}")
    print("-"*80)
    
    all_passed = True
    for r in results:
        if 'error' in r:
            print(f"{r['file']:<45} {'N/A':>8} {'N/A':>8} {'ERROR':>10}")
            all_passed = False
        else:
            notes_str = "✓" if r['notes_ok'] else "✗"
            timing_str = "✓" if r['timing_ok'] else "✗"
            status = "PASS" if (r['notes_ok'] and r['timing_ok']) else "FAIL"
            print(f"{r['file']:<45} {notes_str:>8} {timing_str:>8} {status:>10}")
            if status == "FAIL":
                all_passed = False
    
    print("\n" + "="*80)
    if all_passed:
        print("✓ ALL FILES PASSED CONSISTENCY CHECKS")
    else:
        print("✗ SOME FILES FAILED CONSISTENCY CHECKS")
    print("="*80)

Multi-file consistency check:

File                                             Notes   Timing     Status
--------------------------------------------------------------------------------
Rhapsody - When Demons Awake (solo).gp3.tokens.txt        ✓        ✓       PASS
Knopfler, Mark - Sailing To Philadelphia.gp3.tokens.txt        ✓        ✓       PASS
Guitar Player Lia%0es - Licks de Frank Gambale.gp3.tokens.txt        ✓        ✓       PASS

✓ ALL FILES PASSED CONSISTENCY CHECKS


## 10. Summary Report

In [11]:
if sample_files:
    print("\n" + "="*80)
    print("CONSISTENCY CHECK SUMMARY")
    print("="*80)
    print("\nChecks performed:")
    print("  1. Note count: Raw tokens → NOTE_ON → TAB events")
    print("  2. Timing: Wait tokens → TIME_SHIFT events")
    print("  3. Note details: String, fret, pitch calculations")
    print("  4. Sequence alignment: Input + TAB = Output")
    print("  5. Metadata: Tempo, downtune extraction")
    print("\nConclusion:")
    print("  The dadagp_parser.py module correctly transforms DadaGP tokens")
    print("  into NOTE_ON/NOTE_OFF/TIME_SHIFT/TAB events while preserving:")
    print("    - Note counts")
    print("    - Timing information")
    print("    - String/fret/pitch relationships")
    print("    - Sequence alignment (output = input + TAB)")
    print("\nThis format matches the Fretting-Transformer paper's v3 encoding:")
    print("  Input:  NOTE_ON + NOTE_OFF + TIME_SHIFT")
    print("  Output: NOTE_ON + NOTE_OFF + TIME_SHIFT + TAB")
    print("="*80)


CONSISTENCY CHECK SUMMARY

Checks performed:
  1. Note count: Raw tokens → NOTE_ON → TAB events
  2. Timing: Wait tokens → TIME_SHIFT events
  3. Note details: String, fret, pitch calculations
  4. Sequence alignment: Input + TAB = Output
  5. Metadata: Tempo, downtune extraction

Conclusion:
  The dadagp_parser.py module correctly transforms DadaGP tokens
  into NOTE_ON/NOTE_OFF/TIME_SHIFT/TAB events while preserving:
    - Note counts
    - Timing information
    - String/fret/pitch relationships
    - Sequence alignment (output = input + TAB)

This format matches the Fretting-Transformer paper's v3 encoding:
  Input:  NOTE_ON + NOTE_OFF + TIME_SHIFT
  Output: NOTE_ON + NOTE_OFF + TIME_SHIFT + TAB


## 11. Visual Comparison: DadaGP Tokens vs Parsed Events

Render both DadaGP tokens and parsed events side-by-side to verify they look identical.

In [12]:
from src.visualization import (
    render_dadagp_tokens_as_tablature,
    render_dadagp_tokens_as_notes,
    render_as_tablature,
    render_as_notes
)

print("✓ Visualization functions loaded")

✓ Visualization functions loaded


### 11.1 Tablature Comparison

In [13]:
if raw_tokens and output_events:
    print("TABLATURE COMPARISON (First 3 bars)")
    print("="*80)
    print("\nFROM DADAGP TOKENS:")
    print("-"*80)
    print(render_dadagp_tokens_as_tablature(raw_tokens, max_bars=20, bars_per_row=4))
    
    print("\n\nFROM PARSED EVENTS:")
    print("-"*80)
    print(render_as_tablature(output_events, max_bars=20, bars_per_row=4))
    
    print("\n" + "="*80)
    print("If both look identical, the rendering is consistent!")
    print("="*80)

TABLATURE COMPARISON (First 3 bars)

FROM DADAGP TOKENS:
--------------------------------------------------------------------------------
Guitar Tablature (Standard Tuning: EADGBE) - from DadaGP tokens
e|12--17--16--17------------------|--------------------------------|--------------------------------|--------------------------------|
B|----------------13--17--15--17--|15--18--17--18--13--17--15--17--|13--17--15--17------------------|--------------------------------|
G|--------------------------------|--------------------------------|----------------13--16--14--16--|--------------------------------|
D|--------------------------------|--------------------------------|--------------------------------|12--15--14--15------------------|
A|--------------------------------|--------------------------------|--------------------------------|----------------12--15--14--15--|
E|--------------------------------|--------------------------------|--------------------------------|----------------------

### 11.2 Note Notation Comparison

In [14]:
if raw_tokens and input_events:
    print("NOTE NOTATION COMPARISON (First 3 bars)")
    print("="*80)
    print("\nFROM DADAGP TOKENS:")
    print("-"*80)
    print(render_dadagp_tokens_as_notes(raw_tokens, max_bars=3, bars_per_row=3))
    
    print("\n\nFROM PARSED EVENTS:")
    print("-"*80)
    print(render_as_notes(input_events, max_bars=3, bars_per_row=3))
    
    print("\n" + "="*80)
    print("If both look identical, the rendering is consistent!")
    print("="*80)

NOTE NOTATION COMPARISON (First 3 bars)

FROM DADAGP TOKENS:
--------------------------------------------------------------------------------
Note Notation (like tablature, showing note names) - from DadaGP tokens
 |E3--A3--G#3-A3--A#3-D4--C4--D4--|C4--D#4-D4--D#4-A#3-D4--C4--D4--|A#3-D4--C4--D4--D#4-F#4-E4--F#4-|
 |             Bar 1              |             Bar 2              |             Bar 3              |

... (23 more bars not shown)


FROM PARSED EVENTS:
--------------------------------------------------------------------------------
Note Notation (like tablature, showing note names)
 |E3--A3--G#3-A3--A#3-D4--C4--D4--|C4--D#4-D4--D#4-A#3-D4--C4--D4--|A#3-D4--C4--D4--D#4-F#4-E4--F#4-|
 |             Bar 1              |             Bar 2              |             Bar 3              |

... (23 more bars not shown)

If both look identical, the rendering is consistent!


### 11.3 Multi-Row Display (6 bars per row)

In [15]:
if raw_tokens:
    print("TABLATURE WITH 6 BARS PER ROW")
    print("="*80)
    print(render_dadagp_tokens_as_tablature(raw_tokens, max_bars=12, bars_per_row=6, chars_per_beat=6))

TABLATURE WITH 6 BARS PER ROW
Guitar Tablature (Standard Tuning: EADGBE) - from DadaGP tokens
e|12-17-16-17-------------|------------------------|------------------------|------------------------|------------------------|------------------------|
B|------------13-17-15-17-|15-18-17-18-13-17-15-17-|13-17-15-17-------------|------------------------|------------------------|------------------------|
G|------------------------|------------------------|------------13-16-14-16-|------------------------|------------------------|------------------------|
D|------------------------|------------------------|------------------------|12-15-14-15-------------|------------------------|------------------------|
A|------------------------|------------------------|------------------------|------------12-15-14-15-|11----------------------|------------------------|
E|------------------------|------------------------|------------------------|------------------------|------------------------|--------------